# Обработка документов без веб-интерфейса

Запускайте ячейки сверху вниз в Jupyter или VS Code. Notebook вызывает тот же CLI
DocPipe, который использует общий конвейер экспорта с веб-сервисом. Сервер и HTTP
не нужны. Поддерживаются PDF и изображения, один файл или папка.

1. Проверьте путь проекта и Python в первой ячейке.
2. Укажите документы в `INPUTS` во второй ячейке.
3. Выполните проверку окружения, затем ячейку обработки.
4. Результаты, журналы и архивы появятся в `reports/notebook-runs/`.

По умолчанию используется локальный OCR на CPU, перепроверка выключена.
Зависимости и модели должны быть подготовлены заранее. Этот notebook не выполняет
установку пакетов, загрузку весов или обновление Docling.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import deque
import csv
import fnmatch
import html
import json
import os
import signal
import subprocess
import sys
import time
import uuid
import zipfile
from IPython.display import display, HTML, Markdown, FileLink

# Если notebook открыт за пределами проекта, задайте полный путь здесь.
PROJECT_HINT = Path(r"C:\Users\andre\Desktop\test_OCR")

def is_project(path):
    return (path / "pyproject.toml").is_file() and (path / "src/docpipe/cli.py").is_file()

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, PROJECT_HINT]
                     if is_project(p)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Укажите папку проекта в PROJECT_HINT")
PROJECT_ROOT = PROJECT_ROOT.resolve()

# Обработка запускается этим Python, независимо от окружения ядра Jupyter.
venv_python = PROJECT_ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
PYTHON = venv_python if venv_python.is_file() else Path(sys.executable)
print("Проект:", PROJECT_ROOT)
print("Python обработки:", PYTHON)


## Настройки

В `INPUTS` можно указать несколько файлов и/или папок. Если рядом со сканом лежит
эталон, укажите именно файл скана или задайте `PATTERNS = ["*_no_text.pdf"]`.
Папка `data/input` создаётся для удобства: положите туда документы либо измените путь.

`REVIEW_MODE`: `none` — обычный OCR; `nemotron` — локальная перепроверка Nemotron;
`glm` — существующая перепроверка через Ollama (ей нужен запущенный локальный сервис).
Перепроверка Nemotron на CPU может занимать десятки минут. Частичный результат
перепроверки и необходимость проверки показываются в итоговой таблице.


In [ ]:
INPUTS = [PROJECT_ROOT / "data" / "input"]
# Пример для одного скана:
# INPUTS = [Path(r"C:\папка\документ_no_text.pdf")]

RECURSIVE = False
PATTERNS = ["*.pdf", "*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp", "*.webp"]
REVIEW_MODE = "none"            # none / nemotron / glm
CONFIG_FILE = None              # Дополнительный YAML: Path(...), либо None
EXPORT_DOCX = False
EXPORT_SEARCHABLE_PDF = False
CREATE_ZIP = True
STOP_ON_ERROR = False
SHOW_FULL_LOG = False
CPU_THREADS = 6
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "notebook-runs"

(PROJECT_ROOT / "data" / "input").mkdir(parents=True, exist_ok=True)

if REVIEW_MODE not in {"none", "nemotron", "glm"}:
    raise ValueError("REVIEW_MODE: none, nemotron или glm")
if not isinstance(CPU_THREADS, int) or CPU_THREADS < 1:
    raise ValueError("CPU_THREADS должен быть положительным целым числом")
mode_configs = {"nemotron": "configs/nemotron-review.yaml", "glm": "configs/glm-ocr-review.yaml"}
config_path = Path(CONFIG_FILE) if CONFIG_FILE is not None else (
    PROJECT_ROOT / mode_configs[REVIEW_MODE] if REVIEW_MODE in mode_configs else None
)
if config_path is not None:
    config_path = (PROJECT_ROOT / config_path).resolve() if not config_path.is_absolute() else config_path.resolve()
    if not config_path.is_file():
        raise FileNotFoundError(config_path)

def find_inputs(items):
    found = {}
    for item in items:
        path = Path(item).expanduser()
        path = (PROJECT_ROOT / path).resolve() if not path.is_absolute() else path.resolve()
        if path.is_file():
            found[os.path.normcase(str(path))] = path
        elif path.is_dir():
            candidates = path.rglob("*") if RECURSIVE else path.iterdir()
            for candidate in candidates:
                if candidate.is_file() and any(fnmatch.fnmatchcase(candidate.name.casefold(), pattern.casefold()) for pattern in PATTERNS):
                    resolved = candidate.resolve()
                    # Не включаем результаты предыдущих запусков в рекурсивный поиск.
                    if not resolved.is_relative_to(OUTPUT_ROOT.resolve()):
                        found[os.path.normcase(str(resolved))] = resolved
        else:
            raise FileNotFoundError(f"Входной путь не найден: {path}")
    return sorted(found.values(), key=lambda p: str(p).casefold())

FILES = find_inputs(INPUTS)
if not FILES:
    raise ValueError("Документы не найдены. Положите файлы в data/input или измените INPUTS/PATTERNS.")
print(f"Выбрано документов: {len(FILES)}; перепроверка: {REVIEW_MODE}; устройство: CPU")
for index, path in enumerate(FILES, 1):
    print(f"{index:03}: {path}")


## Проверка окружения

Выполняется в выбранном `PYTHON`, не в Python ядра Jupyter. Настройки этого запуска
имеют приоритет над переменными `DOCPIPE_` в окружении. Пути к моделям и артефактам,
которые вы настроили через окружение, сохраняются.


In [ ]:
env = os.environ.copy()
env.update({
    "PYTHONPATH": str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", ""),
    "PYTHONUNBUFFERED": "1", "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
    "CUDA_VISIBLE_DEVICES": "-1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "OMP_NUM_THREADS": str(CPU_THREADS), "MKL_NUM_THREADS": str(CPU_THREADS),
    "DOCPIPE_OFFLINE": "true", "DOCPIPE_GPU": "false",
    "DOCPIPE_REVIEW__LLM_ENABLED": str(REVIEW_MODE != "none").lower(),
    "DOCPIPE_REVIEW__VISUAL_ENABLED": "false",
    "DOCPIPE_REVIEW__PROVIDER": {"none": "aitunnel", "nemotron": "nemotron", "glm": "ollama"}[REVIEW_MODE],
    "DOCPIPE_REVIEW__OLLAMA_MODE": "reread",
    "DOCPIPE_REVIEW__NEMOTRON_DEVICE": "cpu",
    "DOCPIPE_REVIEW__TIMEOUT_SECONDS": "120", "DOCPIPE_REVIEW__STAGE_TIMEOUT_SECONDS": "900",
    "DOCPIPE_REVIEW__MAX_CALLS": "300",
    "DOCPIPE_RENDER__DOCX_ENABLED": str(EXPORT_DOCX).lower(),
})
temp_dir = PROJECT_ROOT / "reports" / "notebook-tmp"
temp_dir.mkdir(parents=True, exist_ok=True)
env["TEMP"] = env["TMP"] = env["TMPDIR"] = str(temp_dir)
env["HF_MODULES_CACHE"] = str(PROJECT_ROOT / "models" / "hf-modules")
config_args = ["--config", str(config_path)] if config_path is not None else []

doctor = subprocess.run([str(PYTHON), "-m", "docpipe", "doctor", *config_args],
                        cwd=PROJECT_ROOT, env=env, capture_output=True,
                        text=True, encoding="utf-8", errors="replace", timeout=120)
if doctor.returncode:
    print(doctor.stdout, doctor.stderr)
    raise RuntimeError("Проверка окружения не пройдена. Проверьте Python и установленные зависимости.")
health = json.loads(doctor.stdout)
for check in health["checks"]:
    if check["status"] != "PASS" and check["check"] != "api_auth":
        print(check["status"], check["check"], check["detail"])
probe = subprocess.run([str(PYTHON), "-c",
    "from importlib.metadata import version; print('Docling:', version('docling'))"],
    cwd=PROJECT_ROOT, env=env, capture_output=True, text=True, encoding="utf-8", timeout=120)
if probe.returncode:
    raise RuntimeError("В выбранном Python отсутствует Docling. Установите зависимости проекта [docling].")
print(probe.stdout.strip())
if REVIEW_MODE == "nemotron":
    # Проверяется фактический путь из объединённой конфигурации YAML + окружение.
    model_probe = subprocess.run([str(PYTHON), "-c",
        "from pathlib import Path; import sys; from docpipe.config import load_config; "
        "c=load_config(Path(sys.argv[1]) if sys.argv[1] else None); "
        "p=c.review.nemotron_model.resolve(); "
        "assert (p/'model.safetensors').is_file(), f'Веса Nemotron не найдены: {p}'; "
        "import timm, transformers, bs4, pymorphy3; print('Nemotron: зависимости и веса найдены')",
        str(config_path) if config_path else ""], cwd=PROJECT_ROOT, env=env,
        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=180)
    if model_probe.returncode:
        print(model_probe.stderr)
        raise RuntimeError("Не выполнены условия запуска Nemotron; см. docs/nemotron-review.md")
    print(model_probe.stdout.strip())
print("Окружение готово.")


## Обработка

Каждый запуск создаёт новую папку, каждый документ — свой подкаталог.
Создаются JSON, Markdown, HTML, PDF с восстановленным расположением текста и
потоковый PDF; DOCX и поисковый PDF включаются настройками выше.
Параметр `--outputs` существующего CLI не отключает обязательные базовые экспорты.

Полный журнал пишется в `processing.log`. Ошибка одного документа по умолчанию
не останавливает остальные. Кнопка **Interrupt / Stop** прекращает текущий
дочерний процесс и сохраняет сводку уже обработанных документов.


In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}

def stop_child(process):
    if process.poll() is not None:
        return
    if os.name == "nt":
        # У Windows venv есть процесс-лаунчер: останавливаем и его дочерний Python.
        subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    else:
        os.killpg(process.pid, signal.SIGTERM)
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        if os.name != "nt":
            os.killpg(process.pid, signal.SIGKILL)
        else:
            process.kill()
        process.wait()

def process_document(source, out):
    out.mkdir(parents=True, exist_ok=False)
    requested = ["md", "html", "pdf-flow", "pdf-positional"]
    if EXPORT_SEARCHABLE_PDF:
        requested.append("pdf-searchable")
    command = [str(PYTHON), "-m", "docpipe", "parse", str(source),
               "--engine", "docling", "--out", str(out), *config_args,
               "--outputs", *requested]
    started = time.perf_counter()
    tail = deque(maxlen=20)
    options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {"start_new_session": True}
    with (out / "processing.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=PROJECT_ROOT, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1, **options)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                tail.append(line.rstrip())
                if SHOW_FULL_LOG or "OCR pages:" in line:
                    print(line.rstrip(), flush=True)
            exit_code = process.wait()
        except BaseException:
            stop_child(process)
            raise
        finally:
            process.stdout.close()

    metrics = read_json(out / "metrics.json")
    verification = read_json(out / "verification.json")
    review = read_json(out / "review-before.json")
    if exit_code != 0:
        state = "verification_failed" if (out / "document.json").is_file() else "failed"
    elif not metrics or not (out / "document.json").is_file():
        state = "missing_results"
    elif metrics.get("pages_failed", 0) or not metrics.get("text_characters", 0):
        state = "ocr_failed"
    elif review.get("status") in {"partial", "skipped"} or metrics.get("review_required"):
        state = "needs_review"
    else:
        state = "completed"
    record = dict(source=str(source), output=str(out), status=state,
                  seconds=round(time.perf_counter() - started, 2), exit_code=exit_code,
                  pages=metrics.get("pages"), pages_failed=metrics.get("pages_failed"),
                  review_required=metrics.get("review_required"), review_status=review.get("status"),
                  verification=verification.get("result"), archive=None)
    if CREATE_ZIP and (out / "document.json").is_file():
        archive_path = out.with_suffix(".zip")
        with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(out.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(out).as_posix())
        record["archive"] = str(archive_path)
    if state not in {"completed", "needs_review"}:
        print("Последние строки журнала:\n" + "\n".join(tail))
    return record

def save_manifest(records, run_dir):
    payload = dict(run_id=run_dir.name, project=str(PROJECT_ROOT), python=str(PYTHON),
                   review_mode=REVIEW_MODE, device="cpu", documents=records)
    target = run_dir / "manifest.json"
    temp = target.with_suffix(".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(target)
    if records:
        with (run_dir / "summary.csv").open("w", encoding="utf-8-sig", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(records[0]))
            writer.writeheader()
            writer.writerows(records)


In [ ]:
# Эта ячейка запускает обработку. Повторное выполнение создаёт новый запуск.
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
records = []
save_manifest(records, RUN_DIR)
print("Результаты:", RUN_DIR)

for index, source in enumerate(FILES, 1):
    out = RUN_DIR / f"document_{index:03d}"
    print(f"\n[{index}/{len(FILES)}] {source.name}", flush=True)
    try:
        record = process_document(source, out)
    except (Exception, KeyboardInterrupt) as exc:
        record = dict(source=str(source), output=str(out),
                      status="interrupted" if isinstance(exc, KeyboardInterrupt) else "failed",
                      seconds=None, exit_code=None, pages=None, pages_failed=None,
                      review_required=True, review_status=None, verification=None, archive=None)
        (RUN_DIR / f"error_{index:03d}.txt").write_text(f"{type(exc).__name__}: {exc}", encoding="utf-8")
        records.append(record)
        save_manifest(records, RUN_DIR)
        if isinstance(exc, KeyboardInterrupt) or STOP_ON_ERROR:
            raise
        print(f"Ошибка: {type(exc).__name__}: {exc}")
        continue
    records.append(record)
    save_manifest(records, RUN_DIR)
    print(f"{record['status']}; страниц: {record['pages']}; {record['seconds']} с")
    if STOP_ON_ERROR and record["status"] not in {"completed", "needs_review"}:
        raise RuntimeError(f"Обработка остановлена: {source.name}; см. processing.log")

print(f"\nОбработано {len(records)} документов. Сводка: {RUN_DIR / 'manifest.json'}")


## Результаты

`completed` — обработка завершена; `needs_review` — результаты сохранены, есть
замечания или частичная перепроверка; остальные статусы требуют изучения журнала.
Успешный запуск сам по себе не измеряет точность OCR: CER требует отдельного эталона.
В каталоге каждого документа находятся `metrics.json`, `verification.json` и
отчёты перепроверки, если она была включена.


In [ ]:
if not records:
    print("Пока нет обработанных документов.")
else:
    headers = ["Документ", "Статус", "Страниц", "Секунд", "Перепроверка", "Верификация"]
    rows = []
    for record in records:
        values = [Path(record["source"]).name, record["status"], record["pages"],
                  record["seconds"], record["review_status"], record["verification"]]
        rows.append("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in values) + "</tr>")
    display(HTML("<table><thead><tr>" + "".join(f"<th>{h}</th>" for h in headers)
                 + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))
    display(FileLink(os.path.relpath(RUN_DIR / "manifest.json", Path.cwd())))
    display(FileLink(os.path.relpath(RUN_DIR / "summary.csv", Path.cwd())))
    for record in records:
        display(HTML("<strong>" + html.escape(Path(record["source"]).name) + "</strong>"))
        out = Path(record["output"])
        for name in ["document-positional.pdf", "document-flow.pdf", "document.html", "document.md",
                     "document.json", "document.docx", "document-searchable.pdf", "processing.log"]:
            path = out / name
            if path.is_file():
                display(FileLink(os.path.relpath(path, Path.cwd())))
        if record["archive"]:
            display(FileLink(os.path.relpath(record["archive"], Path.cwd())))
    print("Если ссылки недоступны из интерфейса Jupyter, откройте папку:", RUN_DIR)
